In [1]:
import os
import glob
import time
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

# ---- project paths -------------------------------------------------------
RAW_DIR = os.path.join("data", "raw")
CSV_DIR = os.path.join("data", "csv")

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)

print("Raw parquet folder :", os.path.abspath(RAW_DIR))
print("Output csv folder  :", os.path.abspath(CSV_DIR))


Raw parquet folder : /Users/s4n/Documents/clg/sem7/bda_capstone2/data/raw
Output csv folder  : /Users/s4n/Documents/clg/sem7/bda_capstone2/data/csv


In [2]:
parquet_files = sorted(glob.glob(os.path.join(RAW_DIR, "*.parquet")))
print(f"Found {len(parquet_files)} parquet file(s):")
for f in parquet_files:
    print(" -", f)

assert len(parquet_files) > 0, (
    "No parquet files found in data/raw/. Either run the download cell above, "
    "or manually copy your *.parquet file(s) into data/raw/ and re-run this cell."
)


Found 3 parquet file(s):
 - data/raw/yellow_tripdata_2026-01.parquet
 - data/raw/yellow_tripdata_2026-02.parquet
 - data/raw/yellow_tripdata_2026-03.parquet


In [3]:
frames = []
for f in parquet_files:
    t0 = time.time()
    df_part = pd.read_parquet(f, engine="pyarrow")
    print(f"{os.path.basename(f):40s} rows={len(df_part):>10,d}  cols={df_part.shape[1]:2d}  "
          f"({time.time() - t0:.1f}s)")
    frames.append(df_part)

df_raw = pd.concat(frames, ignore_index=True)
del frames
print("\nCombined raw shape:", df_raw.shape)
df_raw.head()


yellow_tripdata_2026-01.parquet          rows= 3,724,889  cols=20  (3.3s)
yellow_tripdata_2026-02.parquet          rows= 3,399,866  cols=20  (0.2s)
yellow_tripdata_2026-03.parquet          rows= 3,952,451  cols=20  (0.3s)

Combined raw shape: (11077206, 20)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75


In [4]:
print("Columns in raw data:")
print(list(df_raw.columns))
print("\nDtypes:")
print(df_raw.dtypes)
print("\nShape:", df_raw.shape)
print("\nNull counts:")
print(df_raw.isnull().sum())


Columns in raw data:
['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee']

Dtypes:
VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag                  str
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge

In [5]:
df_raw.describe(include="all").T


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
VendorID,11077206.0,NaN,NaN,NaN,1.876552,1.0,2.0,2.0,2.0,7.0,0.710529
tpep_pickup_datetime,11077206,NaN,NaN,NaN,2026-02-15 16:11:28.931824,2008-12-31 23:03:20,2026-01-24 00:02:16,2026-02-14 21:54:05,2026-03-10 14:52:12,2026-04-01 00:06:25,NaN
tpep_dropoff_datetime,11077206,NaN,NaN,NaN,2026-02-15 16:28:54.533629,2009-01-01 00:07:36,2026-01-24 00:16:40,2026-02-14 22:12:57,2026-03-10 15:13:28,2026-04-02 16:03:47,NaN
passenger_count,8020083.0,NaN,NaN,NaN,1.238929,0.0,1.0,1.0,1.0,9.0,0.646116
trip_distance,11077206.0,NaN,NaN,NaN,6.221664,0.0,1.0,1.8,3.7,328522.2,620.714976
RatecodeID,8020083.0,NaN,NaN,NaN,5.384528,1.0,1.0,1.0,1.0,99.0,20.038324
store_and_fwd_flag,8020083,2,N,8010706,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PULocationID,11077206.0,NaN,NaN,NaN,161.387009,1.0,114.0,161.0,233.0,265.0,66.909506
DOLocationID,11077206.0,NaN,NaN,NaN,160.887215,1.0,107.0,162.0,234.0,265.0,70.9304
payment_type,11077206.0,NaN,NaN,NaN,0.850878,0.0,0.0,1.0,1.0,4.0,0.675896


In [6]:
ATTRIBUTE_DOCS = {
    "VendorID": "Code indicating the TPEP provider that supplied the record (1 = Creative Mobile Technologies, 2 = VeriFone Inc.)",
    "tpep_pickup_datetime": "Date and time when the meter was engaged (trip start)",
    "tpep_dropoff_datetime": "Date and time when the meter was disengaged (trip end)",
    "passenger_count": "Number of passengers in the vehicle (driver-entered value)",
    "trip_distance": "Elapsed trip distance in miles reported by the taximeter",
    "pickup_longitude": "Longitude where the meter was engaged",
    "pickup_latitude": "Latitude where the meter was engaged",
    "dropoff_longitude": "Longitude where the meter was disengaged",
    "dropoff_latitude": "Latitude where the meter was disengaged",
    "PULocationID": "TLC Taxi Zone in which the taximeter was engaged (newer schema)",
    "DOLocationID": "TLC Taxi Zone in which the taximeter was disengaged (newer schema)",
    "RateCodeID": "Final rate code in effect at trip end (1=Standard,2=JFK,3=Newark,4=Nassau/Westchester,5=Negotiated,6=Group ride)",
    "store_and_fwd_flag": "Y if trip record was held in vehicle memory before sending to vendor (no connection), else N",
    "payment_type": "Numeric code for how the passenger paid (1=Credit card,2=Cash,3=No charge,4=Dispute,5=Unknown,6=Voided trip)",
    "fare_amount": "Time-and-distance fare calculated by the meter (USD)",
    "extra": "Miscellaneous extras/surcharges (rush hour, overnight, etc., USD)",
    "mta_tax": "$0.50 MTA tax automatically triggered by the metered rate in use",
    "tip_amount": "Tip amount (auto-populated for credit-card payments)",
    "tolls_amount": "Total amount of tolls paid on the trip",
    "improvement_surcharge": "$0.30 improvement surcharge assessed on hailed trips",
    "total_amount": "Total amount charged to passengers (does not include cash tips)",
}

present_cols = [c for c in df_raw.columns if c in ATTRIBUTE_DOCS]
dict_df = pd.DataFrame(
    {"attribute": present_cols,
     "description": [ATTRIBUTE_DOCS[c] for c in present_cols]}
)
dict_df


,attribute,description
0,VendorID,Code indicating the TPEP provider that supplie...
1,tpep_pickup_datetime,Date and time when the meter was engaged (trip...
2,tpep_dropoff_datetime,Date and time when the meter was disengaged (t...
3,passenger_count,Number of passengers in the vehicle (driver-en...
4,trip_distance,Elapsed trip distance in miles reported by the...
5,store_and_fwd_flag,Y if trip record was held in vehicle memory be...
6,PULocationID,TLC Taxi Zone in which the taximeter was engag...
7,DOLocationID,TLC Taxi Zone in which the taximeter was disen...
8,payment_type,Numeric code for how the passenger paid (1=Cre...
9,fare_amount,Time-and-distance fare calculated by the meter...


In [7]:
df = df_raw.copy()
start_rows = len(df)
report = []  # (rule, rows_removed, rows_remaining)

def log_step(name, before):
    after = len(df)
    report.append((name, before - after, after))
    print(f"{name:45s}  removed={before - after:>10,d}   remaining={after:>10,d}")


In [8]:
# --- 5.1 Drop exact duplicate rows -----------------------------------------
before = len(df)
df = df.drop_duplicates()
log_step("Drop duplicate rows", before)


Drop duplicate rows                            removed=         0   remaining=11,077,206


In [9]:
# --- 5.2 Parse datetimes & drop rows with missing/unparseable timestamps ---
before = len(df)
df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"], errors="coerce")
df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"], errors="coerce")
df = df.dropna(subset=["tpep_pickup_datetime", "tpep_dropoff_datetime"])
log_step("Drop rows with invalid pickup/dropoff datetime", before)


Drop rows with invalid pickup/dropoff datetime  removed=         0   remaining=11,077,206


In [10]:
# --- 5.3 Trip must end after it starts, and last a realistic duration ------
before = len(df)
df["trip_duration_min"] = (
    (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60.0
)
# keep trips between 1 minute and 3 hours (180 min) — filters GPS glitches / stuck meters
df = df[(df["trip_duration_min"] >= 1) & (df["trip_duration_min"] <= 180)]
log_step("Keep trip_duration_min in [1, 180] minutes", before)


Keep trip_duration_min in [1, 180] minutes     removed=   251,006   remaining=10,826,200


In [11]:
# --- 5.4 Passenger count must be realistic for a yellow cab -----------------
before = len(df)
if "passenger_count" in df.columns:
    df["passenger_count"] = pd.to_numeric(df["passenger_count"], errors="coerce")
    df = df.dropna(subset=["passenger_count"])
    df["passenger_count"] = df["passenger_count"].astype(int)
    df = df[(df["passenger_count"] >= 1) & (df["passenger_count"] <= 6)]
log_step("Keep passenger_count in [1, 6]", before)


Keep passenger_count in [1, 6]                 removed= 3,087,738   remaining= 7,738,462


In [12]:
# --- 5.5 Trip distance must be positive and within a sane upper bound -------
before = len(df)
df["trip_distance"] = pd.to_numeric(df["trip_distance"], errors="coerce")
df = df.dropna(subset=["trip_distance"])
df = df[(df["trip_distance"] > 0) & (df["trip_distance"] <= 100)]
log_step("Keep 0 < trip_distance <= 100 miles", before)


Keep 0 < trip_distance <= 100 miles            removed=    23,692   remaining= 7,714,770


In [13]:
# --- 5.6 VendorID must be one of the documented codes -----------------------
before = len(df)
if "VendorID" in df.columns:
    df = df[df["VendorID"].isin([1, 2])]
log_step("Keep VendorID in {1, 2}", before)


Keep VendorID in {1, 2}                        removed=         0   remaining= 7,714,770


In [14]:
# --- 5.7 RateCodeID must be one of the documented codes ---------------------
before = len(df)
if "RateCodeID" in df.columns:
    df["RateCodeID"] = pd.to_numeric(df["RateCodeID"], errors="coerce")
    df = df.dropna(subset=["RateCodeID"])
    df["RateCodeID"] = df["RateCodeID"].astype(int)
    df = df[df["RateCodeID"].isin([1, 2, 3, 4, 5, 6])]
log_step("Keep RateCodeID in {1..6}", before)


Keep RateCodeID in {1..6}                      removed=         0   remaining= 7,714,770


In [15]:
# --- 5.8 store_and_fwd_flag must be Y or N -----------------------------------
before = len(df)
if "store_and_fwd_flag" in df.columns:
    df["store_and_fwd_flag"] = df["store_and_fwd_flag"].astype(str).str.strip().str.upper()
    df = df[df["store_and_fwd_flag"].isin(["Y", "N"])]
log_step("Keep store_and_fwd_flag in {Y, N}", before)


Keep store_and_fwd_flag in {Y, N}              removed=         0   remaining= 7,714,770


In [16]:
# --- 5.9 GPS coordinates must fall within the NYC bounding box --------------
# (only applies to the older lat/long schema; skipped automatically on newer schema)
NYC_LAT_MIN, NYC_LAT_MAX = 40.40, 41.10
NYC_LON_MIN, NYC_LON_MAX = -74.35, -73.60

geo_cols = ["pickup_latitude", "pickup_longitude", "dropoff_latitude", "dropoff_longitude"]
if all(c in df.columns for c in geo_cols):
    before = len(df)
    for c in geo_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=geo_cols)
    df = df[
        df["pickup_latitude"].between(NYC_LAT_MIN, NYC_LAT_MAX)
        & df["dropoff_latitude"].between(NYC_LAT_MIN, NYC_LAT_MAX)
        & df["pickup_longitude"].between(NYC_LON_MIN, NYC_LON_MAX)
        & df["dropoff_longitude"].between(NYC_LON_MIN, NYC_LON_MAX)
        # (0, 0) is a classic "GPS failed to acquire" placeholder
        & ~((df["pickup_latitude"] == 0) & (df["pickup_longitude"] == 0))
        & ~((df["dropoff_latitude"] == 0) & (df["dropoff_longitude"] == 0))
    ]
    log_step("Keep pickup/dropoff coordinates inside NYC bounding box", before)
else:
    print("Lat/long columns not present (newer schema) — skipping geo filter,"
          " will validate PULocationID/DOLocationID instead if present.")
    if "PULocationID" in df.columns and "DOLocationID" in df.columns:
        before = len(df)
        df = df.dropna(subset=["PULocationID", "DOLocationID"])
        log_step("Drop rows with missing PULocationID/DOLocationID", before)


Lat/long columns not present (newer schema) — skipping geo filter, will validate PULocationID/DOLocationID instead if present.
Drop rows with missing PULocationID/DOLocationID  removed=         0   remaining= 7,714,770


In [17]:
# --- 5.10 Fare / money fields must be non-negative (if present) -------------
money_cols = [c for c in ["fare_amount", "extra", "mta_tax", "tip_amount",
                           "tolls_amount", "improvement_surcharge", "total_amount"]
              if c in df.columns]
if money_cols:
    before = len(df)
    for c in money_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=money_cols)
    # negative fares/totals are refunds or data errors for this analysis
    df = df[(df[money_cols] >= 0).all(axis=1)]
    # a fare of 0 with a real trip distance is also implausible — light sanity cap
    if "total_amount" in df.columns:
        df = df[df["total_amount"] <= 500]
    log_step("Keep non-negative fare/money fields (total_amount <= 500)", before)


Keep non-negative fare/money fields (total_amount <= 500)  removed=    76,613   remaining= 7,638,157


In [18]:
print(f"\nTOTAL removed: {start_rows - len(df):,d} of {start_rows:,d} rows "
      f"({(start_rows - len(df)) / start_rows:.1%})")
print(f"Remaining clean rows: {len(df):,d}")

pd.DataFrame(report, columns=["cleaning_step", "rows_removed", "rows_remaining"])



TOTAL removed: 3,439,049 of 11,077,206 rows (31.0%)
Remaining clean rows: 7,638,157


,cleaning_step,rows_removed,rows_remaining
0,Drop duplicate rows,0,11077206
1,Drop rows with invalid pickup/dropoff datetime,0,11077206
2,"Keep trip_duration_min in [1, 180] minutes",251006,10826200
3,"Keep passenger_count in [1, 6]",3087738,7738462
4,Keep 0 < trip_distance <= 100 miles,23692,7714770
5,"Keep VendorID in {1, 2}",0,7714770
6,Keep RateCodeID in {1..6},0,7714770
7,"Keep store_and_fwd_flag in {Y, N}",0,7714770
8,Drop rows with missing PULocationID/DOLocationID,0,7714770
9,Keep non-negative fare/money fields (total_amo...,76613,7638157


In [19]:
df["pickup_date"] = df["tpep_pickup_datetime"].dt.date.astype(str)
df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
df["pickup_day"] = df["tpep_pickup_datetime"].dt.day
df["pickup_month"] = df["tpep_pickup_datetime"].dt.month
df["pickup_year"] = df["tpep_pickup_datetime"].dt.year
df["pickup_dayofweek"] = df["tpep_pickup_datetime"].dt.dayofweek  # 0=Monday
df["pickup_day_name"] = df["tpep_pickup_datetime"].dt.day_name()
df["is_weekend"] = df["pickup_dayofweek"].isin([5, 6]).astype(int)

# trip_duration_min already computed during cleaning; round for readability
df["trip_duration_min"] = df["trip_duration_min"].round(2)

# average speed (mph) — guard against divide-by-zero (already excluded duration==0)
df["avg_speed_mph"] = (df["trip_distance"] / (df["trip_duration_min"] / 60.0)).round(2)
# drop the small number of physically-impossible speeds (sensor noise)
before = len(df)
df = df[(df["avg_speed_mph"] > 0) & (df["avg_speed_mph"] <= 80)]
print(f"Removed {before - len(df):,d} rows with implausible avg_speed_mph (>80 mph)")

# simple fare-per-mile if money columns exist — useful for a Hive/MapReduce metric
if "total_amount" in df.columns:
    df["fare_per_mile"] = (df["total_amount"] / df["trip_distance"]).round(2)

df.head()


Removed 481 rows with implausible avg_speed_mph (>80 mph)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,trip_duration_min,pickup_date,pickup_hour,pickup_day,pickup_month,pickup_year,pickup_dayofweek,pickup_day_name,is_weekend,avg_speed_mph,fare_per_mile
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00,5.55,2026-01-01,0,1,1,2026,3,Thursday,0,10.49,16.35
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75,42.80,2026-01-01,0,1,1,2026,3,Thursday,0,7.82,9.96
5,2,2026-01-01 00:47:11,2026-01-01 01:00:47,2,2.33,1.0,N,144,137,1,14.2,1.00,0.5,4.99,0.0,1.0,24.94,2.5,0.0,0.75,13.60,2026-01-01,0,1,1,2026,3,Thursday,0,10.28,10.70
6,1,2026-01-01 00:17:54,2026-01-01 00:28:32,1,1.30,1.0,N,142,50,2,11.4,4.25,0.5,0.00,0.0,1.0,17.15,2.5,0.0,0.75,10.63,2026-01-01,0,1,1,2026,3,Thursday,0,7.34,13.19
8,2,2026-01-01 00:34:14,2026-01-01 01:11:58,1,5.34,1.0,N,161,45,1,37.3,1.00,0.5,8.61,0.0,1.0,51.66,2.5,0.0,0.75,37.73,2026-01-01,0,1,1,2026,3,Thursday,0,8.49,9.67


In [20]:
print("Final cleaned shape:", df.shape)
df.dtypes


Final cleaned shape: (7637676, 31)


VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                   int64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag                  str
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
cbd_congestion_fee              float64
trip_duration_min               float64
pickup_date                         str
pickup_hour                       int32
pickup_day                        int32
pickup_month                      int32


In [21]:
CLEANED_PATH = os.path.join(CSV_DIR, "yellow_tripdata_cleaned.csv")
df.to_csv(CLEANED_PATH, index=False)
print(f"Wrote {len(df):,d} rows -> {CLEANED_PATH} "
      f"({os.path.getsize(CLEANED_PATH) / 1e6:.1f} MB)")


Wrote 7,637,676 rows -> data/csv/yellow_tripdata_cleaned.csv (1226.4 MB)


In [22]:
SAMPLE_SIZE = 50_000  # >= 5,000 required by the rubric; adjust as needed
SAMPLE_PATH = os.path.join(CSV_DIR, "yellow_tripdata_sample_for_hadoop.csv")

sample_n = min(SAMPLE_SIZE, len(df))
df_sample = df.sample(n=sample_n, random_state=42).sort_values("tpep_pickup_datetime")
df_sample.to_csv(SAMPLE_PATH, index=False)
print(f"Wrote {len(df_sample):,d} rows -> {SAMPLE_PATH} "
      f"({os.path.getsize(SAMPLE_PATH) / 1e6:.1f} MB)")


Wrote 50,000 rows -> data/csv/yellow_tripdata_sample_for_hadoop.csv (8.0 MB)


In [23]:
DICT_PATH = os.path.join(CSV_DIR, "data_dictionary.csv")
dict_df.to_csv(DICT_PATH, index=False)
print(f"Wrote data dictionary -> {DICT_PATH}")


Wrote data dictionary -> data/csv/data_dictionary.csv


In [24]:
print("\nFiles now in data/csv/:")
for f in sorted(glob.glob(os.path.join(CSV_DIR, "*.csv"))):
    print(f" - {f}  ({os.path.getsize(f) / 1e6:.2f} MB)")



Files now in data/csv/:
 - data/csv/data_dictionary.csv  (0.00 MB)
 - data/csv/yellow_tripdata_cleaned.csv  (1226.37 MB)
 - data/csv/yellow_tripdata_sample_for_hadoop.csv  (8.03 MB)


In [25]:
check = pd.read_csv(SAMPLE_PATH)
print("Re-read shape:", check.shape)
assert check.shape == df_sample.shape, "Row/column mismatch after CSV round-trip!"
assert check.isnull().sum().sum() == 0, "Unexpected NaNs introduced by CSV round-trip!"
print("CSV round-trip OK — safe to hand off to HDFS / Hive / MapReduce.")
check.head()


Re-read shape: (50000, 31)
CSV round-trip OK — safe to hand off to HDFS / Hive / MapReduce.


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,trip_duration_min,pickup_date,pickup_hour,pickup_day,pickup_month,pickup_year,pickup_dayofweek,pickup_day_name,is_weekend,avg_speed_mph,fare_per_mile
0,2,2026-01-01 00:08:15,2026-01-01 00:14:44,1,1.74,1.0,N,229,107,1,9.3,1.00,0.5,3.01,0.0,1.0,18.06,2.5,0.0,0.75,6.48,2026-01-01,0,1,1,2026,3,Thursday,0,16.11,10.38
1,1,2026-01-01 00:08:48,2026-01-01 00:14:44,1,1.80,1.0,N,141,236,2,10.7,3.50,0.5,0.00,0.0,1.0,15.70,2.5,0.0,0.00,5.93,2026-01-01,0,1,1,2026,3,Thursday,0,18.21,8.72
2,1,2026-01-01 00:11:15,2026-01-01 00:19:00,3,0.90,1.0,N,79,148,1,9.3,4.25,0.5,3.00,0.0,1.0,18.05,2.5,0.0,0.75,7.75,2026-01-01,0,1,1,2026,3,Thursday,0,6.97,20.06
3,1,2026-01-01 00:11:36,2026-01-01 00:43:21,1,3.60,1.0,N,142,164,1,26.8,4.25,0.5,6.50,0.0,1.0,39.05,2.5,0.0,0.75,31.75,2026-01-01,0,1,1,2026,3,Thursday,0,6.80,10.85
4,2,2026-01-01 00:15:22,2026-01-01 00:22:40,3,1.39,1.0,N,142,237,1,9.3,1.00,0.5,2.86,0.0,1.0,17.16,2.5,0.0,0.00,7.30,2026-01-01,0,1,1,2026,3,Thursday,0,11.42,12.35
